Linearly Constrained Quadratic Optimization in Portfolio Selection
============================

For compatibility and possibility to compare results with NumBook.pdf, I use problem statement and terminology from NumBook.pdf. So the Introduction section (I.) includes extensive quotations fromCourse 589 Lecture Notes. The computations, data analysis and decisions are my own.

A general unconstrained quadratic optimization problem can be formulated as:

$$\min_{\vec{x}∈ℝ^n} f(\vec{x}) = \frac{1}{2}\vec{x}^T*Q*\vec{x} +\vec{c}*\vec{x}$$

where:
 - $\vec{x}$ is an 𝑛-dimensional vector of decision variables.
 - Q is an 𝑛 × 𝑛 symmetric positive definite matrix.
 - $\vec{c}$ is an 𝑛-dimensional vector of coefficients.

The objective is to find the vector x that minimizes the quadratic function 𝑓 (x).

We note that the necessary condition of a minimum is
$$∇f(\vec{x}) = Q*\vec{x} + \vec{c} = 0$$

Thus, unconstrained quadratic optimization is equivalent to solving a linear system with a symmetric, positive definite matrix. Should a solution be found, it is automatically a minimum due to the Second Derivative Criterion.

-----------------

# I. Naïve statement of the problem in Markowitz Portfolio Optimization

In finance, Markowitz’s Modern Portfolio Theory (MPT) provides a framework for constructing portfolios that optimize expected return based on a given level of risk. The risk (**:= variance**) of a portfolio composed of multiple assets
is a quadratic function of the asset weights.

## I.1 Portfolio Variance as a Quadratic Function

Consider a portfolio of 𝑛 assets with weights
$\vec{w} = \begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_n
\end{bmatrix}$ which must sum up to 1.

The portfolio variance $𝜎^2$ is given by:
$$𝜎^2 = \vec{w}^T * Σ * \vec{w}$$ where $Σ$ is the covariance matrix of asset returns.

## I.2 Linearly Constrained Optimization Problem

The portfolio variance minimization problem is formulated as:

- minimize: $𝑓 (\vec{w}) = \frac{1}{2}\vec{w}^⊤*Σ*\vec{w}$

- subject to: $\vec{𝟙}^⊤*\vec{w} = 1.$

- Here $\vec{𝟙} := \begin{bmatrix}
1_1 \\
1_2 \\
\vdots \\
1_n
\end{bmatrix}$

## I.3 Another statement of Optimization Problem

Suppose now that we desire an expected return 𝑑 of 10% from our portfolio. Find weight vector $\vec{w}$ such that the portfolio has the expected return 𝑑 = 0.1 and minimum variance.

This leads to a constraint:
$$\vec{r}^⊤*\vec{w} = 𝑑$$

where 𝑑 = 0.1, $\vec{r}$  is return vector.

The new problem is

- minimize: $𝑓 (\vec{w}) = \frac{1}{2}\vec{w}^⊤*Σ*\vec{w}$
- subject to: $\vec{r}^T*\vec{w} = 𝑑$
- and subject to: $\vec{𝟙}^⊤*\vec{w} = 1.$

Again, we minimize a quadratic function, but this time subject to two linear constraints. The constraints are independent unless returns of all assets are the same (thus making the problem trivial). Therefore, we can eliminate two variables from the quadratic form, thus leaving an unconstrained problem.

---

# II. Why Markowitz Portfolio Theory often "fails" in practice

!! По идее это уже выводы, которые нужно вставлять после анализа данных!!

1. **Overly restrictive assumptions**

   * Asset returns are assumed to be **Gaussian**, while empirical distributions exhibit heavy tails and skewness.
   * The covariance structure is assumed stationary, yet correlations shift dramatically in times of market stress.

2. **Estimation sensitivity**

   * Optimal portfolio weights rely on the inverse of the covariance matrix $\Sigma^{-1}$. Even small estimation errors in variances or covariances are amplified, leading to unstable weights.
   * The issue is exacerbated when the number of assets $N$ is comparable to or exceeds the number of observations $T$.

3. **Implementation constraints**

   * The unconstrained optimization frequently produces extreme long–short allocations, which are practically unimplementable and highly unstable.

## II.1. Problems specifically with the covariance matrix

1. **Estimation error**

   * The sample covariance estimator

     $$
     \hat{\Sigma} = \frac{1}{T-1} \sum_{t=1}^T (r_t - \bar{r})(r_t - \bar{r})^\top
     $$

     has high variance, particularly when $T$ is not large compared to $N$.

2. **Ill-conditioning**

   * The inverse $\Sigma^{-1}$ required for computing portfolio weights becomes unstable when eigenvalues of $\Sigma$ span several orders of magnitude.
   * Small eigenvalues ($\lambda_i \approx 0$) amplify noise, effectively causing the optimizer to overfit spurious correlations.

3. **Singularity**

   * If $N > T$, the sample covariance matrix is rank-deficient and not invertible ($\det \hat{\Sigma} = 0$).

## II.3. Indicators of danger in applying Markowitz optimization

1. **Eigenvalue spectrum**

   * Extremely small eigenvalues of $\hat{\Sigma}$ indicate near-singularity.
   * A spectrum with many tiny eigenvalues and a few dominating ones implies weight instability.

2. **Condition number**

   * Defined as

     $$
     \kappa(\Sigma) = \frac{\lambda_{\max}}{\lambda_{\min}}.
     $$
   * A large $\kappa(\Sigma)$ (say, $10^3$ or higher) implies that $\Sigma^{-1}$ is numerically unstable and that optimal weights are highly sensitive to perturbations.

3. **Dimensional ratio**

   * The ratio $N/T$.
   * If $N/T \approx 1$ or larger, the sample covariance is nearly singular. This is precisely the high-dimensional regime where random matrix theory predicts eigenvalue distributions dominated by noise.


## II.4. Conclusion

Markowitz portfolio optimization is "dangerous" when:

* The covariance matrix is ill-conditioned (large $\kappa(\Sigma)$).
* The eigenvalue spectrum contains very small or negative eigenvalues (in practice, arising from estimation noise).
* The problem is high-dimensional with $N/T$ close to or exceeding 1.

In such cases, the optimal solution overfits sampling noise, leading to portfolios with extreme, unstable allocations. This is why modern practice relies on **regularized covariance estimators** (e.g., Ledoit–Wolf shrinkage, factor models, or random matrix theory–based filtering) rather than naïve sample covariance inversion.

---

# III. Data Preparation and Implementation

## III.1 Detrending Logarithmic Prices

Asset prices often exhibit long-term trends that can obscure short-term fluctuations critical for portfolio optimization. By detrending the logarithmic price series, we eliminate these trends while preserving the properties needed to
compute log-returns.
The logarithmic price series $ln(𝑃_{𝑖,𝑡})$ of an asset 𝑖 at time 𝑡 is modeled as:
$$ln(𝑃_{𝑖,𝑡}) = 𝛽_0 + 𝛽_1*𝑡 + 𝜖_𝑡$$ ,
where:
- $𝛽_0$ is the intercept (initial log-price level),
- $𝛽_1$ is the slope (trend rate),
- $𝜖_𝑡$ is the residual, representing the detrended logarithmic price.

The residual $𝜖_𝑡$ captures **deviations** from the trend. The return between periods is then calculated as:
$$𝑟_{𝑖,𝑡} = 𝜖_𝑡 − 𝜖_{𝑡−1}.$$


## III.2 Return Vector Calculation

Using the detrended logarithmic returns $𝑟_{𝑖,𝑡}$ , the expected return vector 𝜇 is computed as the mean of these returns:
$$𝜇_𝑖 = \frac{1}{𝑇}\sum_{𝑡=1}^{𝑇}𝑟_{𝑖,𝑡}$$ ,
where 𝑇 is the number of time periods.


## III.3 Covariance Matrix Calculation

The covariance matrix Σ measures the relationships between the detrended returns:
$$Σ_{𝑖,𝑗} =\frac{1}{𝑇-1}\sum^{𝑇}_{𝑡=1}(𝑟_{𝑖,𝑡} − 𝜇_𝑖)(𝑟_{𝑗,𝑡} − 𝜇_𝑗).$$
This matrix quantifies the risk and correlation structure among the assets.


## III.4 Annualization of Returns and Covariance Matrix

In financial analysis, it is common practice to annualize returns and risk metrics to standardize them for interpretation and comparison across different timeframes. Annualization transforms daily, weekly, or monthly data into a
yearly equivalent, making it more perceptible and meaningful to investors.

### III.4.1 Annualizing Returns

The return vector 𝜇 computed from daily or other periodic returns can be scaled to reflect annualized returns using the formula:
$$𝜇_{annualized} = 𝑓 ⋅ 𝜇$$,
where:
- 𝑓 is the annualization factor, defined as the number of periods in a year (e.g., 𝑓 = 252 for daily returns, 𝑓 = 52 for weekly returns, 𝑓 = 12 for monthly returns),
- 𝜇 is the mean of periodic returns.

Annualized returns provide a single, consistent measure of performance that investors can use to compare assets or portfolios.


### III.4.2 Annualizing the Covariance Matrix

The covariance matrix Σ measures the relationships between asset returns over the given period. To align it with annualized returns, it must be scaled by the square of the annualization factor:
$$Σ_{annualized} = 𝑓^2 ⋅ Σ$$,
where:
- Σ is the covariance matrix computed from periodic returns,
- $𝑓^2$ adjusts for the scaling of variance (quadratic in time).

This adjustment ensures that the risk measures derived from Σ, such as portfolio variance, correspond to annualized returns.

### III.4.3 Correlation Matrix

The correlation matrix quantifies the linear relationships between asset returns, providing a scale-invariant measure of how the returns of two assets move relative to one another. For a set of 𝑛 assets, the correlation matrix **R** is
computed as:

$$𝑅_{𝑖,𝑗} = \frac{Cov(𝑟_𝑖, 𝑟_𝑗)}{𝜎_𝑖𝜎_𝑗}$$
,
where:

- $Cov(𝑟_𝑖, 𝑟_𝑗)$ is the covariance between the returns of assets 𝑖 and 𝑗,
- $𝜎_𝑖$ and $𝜎_𝑗$ are the standard deviations of the returns of assets 𝑖 and 𝑗.

The entries of the correlation matrix range from −1 (perfect negative correlation) to 1 (perfect positive correlation), with 0 indicating no linear relationship. The diagonal elements of the matrix are always 1, as each asset is
perfectly correlated with itself.

Rationale: The correlation matrix is independent of the time scale of the data, making it a fundamental tool for understanding diversification potential and the interdependencies between assets in a portfolio.

## III.4.4 Interpretation of Annualized Metrics

Annualized returns and covariance matrix values have the following characteristics:
- **Annualized Returns ($𝜇_{annualized}$)**: Represent the average return expected over a year. For example, an annualized return of 0.05 corresponds to a 5% yearly gain.
- **Annualized Covariance ($Σ_{annualized}$)**: Captures the annualized variability and correlations between asset returns, making portfolio risk measures like standard deviation (volatility) interpretable in yearly terms.


### III.5  Daily Data

For daily returns with 252 trading days per year:
$$𝜇_{annualized} = 252 ⋅ 𝜇, Σ_{annualized} = 252^2 ⋅ Σ$$.
These formulas standardize daily estimates to yearly equivalents.


### III.6 Practical Considerations

- **Frequency of Data**: Ensure that the correct annualization factor 𝑓 is used for the data frequency.
- **Assumptions**: Annualization assumes that the statistical properties of the returns remain stable over time (e.g., stationarity, consistent market conditions).
- **Interpretability**: Annualization provides an intuitive scale for assessing the expected performance and risk of assets and portfolios, aiding decision-making for investors.

Annualization is an essential step in portfolio analysis, bridging the gap between short-term data and long-term investment horizons.
